In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
from datetime import date
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

BASE_DIR     = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
HIST_PATH    = os.path.join("C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/2025/Outputs/Output4/Indicadores/precipitacion diaria/")

os.chdir(BASE_DIR)
print("Base dir :", BASE_DIR)
print("Histórico:", HIST_PATH)


Base dir : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Histórico: C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/2025/Outputs/Output4/Indicadores/precipitacion diaria/


## Paso 1 – Descarga datos del día más reciente (API IDEAM)

In [2]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)
fecha_mapa = client.get(DATASET_ID, select="max(fechaobservacion)")[0]["max_fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)

records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros...")
client.close()
print(f"Total: {len(records):,}")


Última fecha disponible: 2026-03-29


  100,000 registros...


  164,359 registros...


Total: 164,359


## Paso 2 – Agregar lecturas 10-min a nivel diario por estación

In [3]:
df_api = pd.DataFrame.from_records(records)

for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]

df_dia = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_diaria = ("valorobservado", "sum"),
        precip_max_10min   = ("valorobservado", "max"),
        precip_min_10min   = ("valorobservado", "min"),
        precip_media_10min = ("valorobservado", "mean"),
        n_lecturas         = ("valorobservado", "count"),
    )
)
del df_api

print(f"Estaciones activas ({fecha_mapa}): {len(df_dia):,}")
df_dia["fecha"]=date.today().strftime("%Y-%m-%d")
df_dia.head()


Estaciones activas (2026-03-29): 619


,codigoestacion,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,precip_acum_diaria,precip_max_10min,precip_min_10min,precip_media_10min,n_lecturas,fecha
0,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862000,-76.152056,0.0,0.0,0.0,0.000000,144,2026-03-30
1,0011035010,LLORO,CHOCO,LLORÓ,ATRATO - DARIÉN,5.499000,-76.539000,28.3,6.2,0.0,0.199296,142,2026-03-30
2,0011045010,AEROPUERTO EL CARAÑO,CHOCO,QUIBDÓ,ATRATO - DARIÉN,5.690556,-76.643778,0.0,0.0,0.0,0.000000,863,2026-03-30
3,0011047020,QUIBDO,CHOCO,QUIBDÓ,ATRATO - DARIÉN,5.697778,-76.662222,0.8,0.5,0.0,0.005556,144,2026-03-30
4,0011057020,SAN ANTONIO PADUA,ANTIOQUIA,VIGIA DEL FUERTE,ATRATO - DARIÉN,6.286667,-76.761806,0.0,0.0,0.0,0.000000,144,2026-03-30


In [4]:
#df_dia.to_excel(HIST_PATH, index=False)

## Paso 3 – Concatenar con histórico y guardar

In [5]:
HIST_PATH = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx"

COLS = [
    "codigoestacion", "nombreestacion", "departamento", "municipio",
    "zonahidrografica", "latitud", "longitud", "fecha",
    "precip_acum_diaria", "precip_max_10min", "precip_min_10min",
    "precip_media_10min", "n_lecturas",
]

def normalizar(df):
    df = df.copy()
    df.columns = df.columns.str.lower()
    for col in COLS:
        if col not in df.columns:
            df[col] = np.nan
    return df[COLS]

df_hist = normalizar(pd.read_excel(HIST_PATH))

df_nuevo = normalizar(df_dia.copy())
df_nuevo["fecha"] = pd.to_datetime(fecha_mapa)

df_daily = (
    pd.concat([df_hist, df_nuevo], ignore_index=True)
    .drop_duplicates(subset=["codigoestacion", "fecha"], keep="last")
    .sort_values(["codigoestacion", "fecha"])
    .reset_index(drop=True)
)

df_daily.to_excel(HIST_PATH, index=False)

print(f"Guardado: {HIST_PATH}")
print(f"Registros anteriores : {len(df_hist):,}")
print(f"Registros nuevos     : {len(df_nuevo):,}")
print(f"Total                : {len(df_daily):,}")


Guardado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx
Registros anteriores : 2,967
Registros nuevos     : 619
Total                : 3,586
